In [1]:
import polars as pl
simple = pl.read_csv("data/simple_rules_abbreviations.csv")
simple = simple.sort(by="abbr_parts", descending=True)
volume_specific = [simple.filter(pl.col("volumes").str.contains(rf"{volume}|\*")) for volume in range(10)]

In [2]:
simple

abbreviation,expansion,translation,stem,declension,volumes,abbr_parts
str,str,str,str,str,str,i64
"""benef. c. c. vel s. c.""","""beneficium cum cura vel sine c…","""Pfründe mit oder ohne Seelsorg…",null,null,"""7|8""",6
"""s. e. s. o. n. c.""","""sed eius superveniente obitu n…",null,null,null,"""5""",6
"""ben. c. v. s. c.""","""beneficium cum vel sine cura""","""Pfründe mit oder ohne Seelsorg…",null,null,"""1""",5
"""benef. c. v. s. c.""","""beneficium cum vel sine cura""","""Pfründe mit oder ohne Seelsorg…",null,null,"""6""",5
"""c. c. vel s. c.""","""cum cura vel sine cura""","""mit oder ohne Seelsorge""",null,null,"""7|8""",5
…,…,…,…,…,…,…
"""vic.""","""vicarius""","""Vikar, Stellvertreter""","""vicari -i""","""o""","""2|3|4|5|6|7|8|9""",1
"""vicar.""","""vicaria""","""Vikarie, Meßpriesterstelle""","""vicari -(a)e""","""a""","""2|3|4|5|6|7|8|9""",1
"""virg.""","""virgo""","""Jungfrau""","""virgin -is""","""konsonantisch""","""1|2|4|5|6|7|8|9""",1


In [3]:
volume_specific[2]

abbreviation,expansion,translation,stem,declension,volumes,abbr_parts
str,str,str,str,str,str,i64
"""abb. et. conv.""","""abbas et conventus""","""Abt und Konvent""","""abbat -is; convent -us""","""konsonantisch/u""","""2|3|4|5|6|8|9""",3
"""aep. et capit.""","""archiepiscopus, prepositus, de…","""Erzbischof, Propst, Dekan und …",null,null,"""2|3|4|5|9""",3
"""dec. et cap.""","""decanus, capitulum et singuli …","""Dekan, Kapitel und die einzeln…",null,null,"""1|2|4""",3
"""dec. et capit.""","""decanus, capitulum et singuli …","""Dekan, Kapitel und die einzeln…",null,null,"""2|3|4|5|6|7|8|9""",3
"""ep. et capit.""","""episcopus et capitulum""","""Bischof und Kapitel""",null,null,"""2|3|4|5|7|8|9""",3
…,…,…,…,…,…,…
"""vic.""","""vicarius""","""Vikar, Stellvertreter""","""vicari -i""","""o""","""2|3|4|5|6|7|8|9""",1
"""vicar.""","""vicaria""","""Vikarie, Meßpriesterstelle""","""vicari -(a)e""","""a""","""2|3|4|5|6|7|8|9""",1
"""virg.""","""virgo""","""Jungfrau""","""virgin -is""","""konsonantisch""","""1|2|4|5|6|7|8|9""",1


In [4]:
rg = pl.read_csv("data/rg.csv")
rg = rg.sort(by="id_RG_all")

In [5]:
rg

volume,nr_RG,header_no_tags,regest_no_tags,id_RG_all
i64,i64,str,str,str
1,1,"""Abardus Alamannuso. s. Ant.""",null,"""10100001-0"""
1,1,null,"""qui portavit litteras d. pape …","""10100001-1"""
1,2,"""Achatius Erhardi Asini de Mont…",null,"""10100002-0"""
1,2,null,"""de benef. ad coll. abb. etc. m…","""10100002-1"""
1,3,"""Achatius Nicolai Wenke de Nisa…",null,"""10100003-0"""
…,…,…,…,…
10,10625,null,"""referentes quod olim Henricus …","""11010625-3"""
10,10625,null,"""referentes quod nonnulli aep.,…","""11010625-4"""
10,10625,null,"""de conserv. 7. mai. 82 S 810 2…","""11010625-5"""


In [6]:
def expand_rg_simple(rg_texts, volume_specific_expansions):
    volumes = {}
    for volume in range(10):
        expansions = volume_specific_expansions[volume]
        texts = rg_texts.filter(pl.col("volume") == volume)
        
        for row in expansions.iter_rows(named=True):
            # since all abbreviations, but no expansions contain `.`, we don't need to worry about expanding what has already been expanded
            texts = texts.with_columns(pl.col("header_no_tags").str.replace_all(row["abbreviation"], row["expansion"], literal=True).alias("header_no_tags"))
            texts = texts.with_columns(pl.col("regest_no_tags").str.replace_all(row["abbreviation"], row["expansion"], literal=True).alias("regest_no_tags"))

        volumes[volume] = texts
    
    return pl.concat(volumes.values())

In [7]:
expanded_texts = expand_rg_simple(rg, volume_specific)

In [8]:
id = "10906306-3"
print(rg.filter(pl.col("id_RG_all") == id).get_column("regest_no_tags").item())
print(expanded_texts.filter(pl.col("id_RG_all") == id).get_column("regest_no_tags").item())

Dom. o. fr. min. e.m. op. Z. Traiect. dioc. de observ. nunc. ap. auct. ad instantiam genitoris Adolphi Gelrie et Juliacen. [ducis] et comitis Zutphanien. erecta sed adhuc non confecta: supplic. d. Adolpho duce de lic. dom. in d. op. de novo erigendi et fratres ad illam transferendi 16. mai. 1470 S 658 282rs.
Dom. o. frater minor e.m. oper Z. Traiect. diocesis de observitium nuncupatus apostolicus auct. ad instantiam genitoris Adolphi Gelrie et Juliacenon [ducis] et comitis Zutphanienon erecta sed adhuc non confecta: supplicentia d. Adolpho duce de licentia dom. in d. oper de novo erigendi et fratres ad illam transferendi 16. mai. 1470 S 658 282rs.


approximating the number of abbreviations by the number of `.`

In [9]:
# filtering for all volumes except 10, because we don't have any rules for volumes 10 yet and are dropping this one when expanding
counts_rg = rg.filter(pl.col("volume") != 10).with_columns(pl.col("header_no_tags").str.count_matches(r"\.").sum().alias("countH"), pl.col("regest_no_tags").str.count_matches(r"\.").sum().alias("countR"))
abbreviations_rg = counts_rg.row(0)[5] + counts_rg.row(0)[6]

counts_expanded = expanded_texts.with_columns(pl.col("header_no_tags").str.count_matches(r"\.").sum().alias("countH"), pl.col("regest_no_tags").str.count_matches(r"\.").sum().alias("countR"))
abbreviations_expanded = counts_expanded.row(0)[5] + counts_expanded.row(0)[6]

print(f"abbreviations in rg: {abbreviations_rg}")
print(f"abbreviations after expanding: {abbreviations_expanded}")

abbreviations in rg: 2483194
abbreviations after expanding: 1067861
